# Dogs vs Cats — Классификация изображений кошек и собак

**Тема:** Бинарная классификация изображений свёрточной нейросетью  
**Инструменты:** `PyTorch`, `torchvision`, `PIL`

---

## О задаче

Даны цветные фотографии — на каждой кошка или собака. Нужно предсказать класс (0 — кошка, 1 — собака). В отличие от MNIST, здесь изображения разного размера и полноцветные, поэтому их нужно привести к единому размеру и преобразовать в тензоры.

**Метка берётся из имени файла:** `dog.123.jpg` → собака, `cat.45.jpg` → кошка.

## 1. Пути и разбиение файлов

Картинки лежат в папках. Список файлов из `train/` делим на обучение и валидацию (80/20).

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Пути к данным (на Kaggle: /kaggle/input/dogs-vs-cats-redux-kernels-edition/...)
image_dir = 'train'
test_dir  = 'test1'

all_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)
print('Train:', len(train_files), '| Val:', len(val_files))

## 2. `Dataset` и преобразования

Каждое изображение приводим к `128×128` и переводим в тензор. На обучении добавляем случайное горизонтальное отражение — простая **аугментация**, которая увеличивает разнообразие данных и уменьшает переобучение.

In [ ]:
class CatDogDataset(Dataset):
    def __init__(self, filenames, directory, is_test=False, transform=None):
        self.filenames = filenames
        self.dir = directory
        self.is_test = is_test
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        image = Image.open(os.path.join(self.dir, name)).convert('RGB')
        if self.transform:
            image = self.transform(image)
        if not self.is_test:
            label = 1 if 'dog' in name.lower() else 0
            return image, torch.tensor(label, dtype=torch.long)
        return image

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_loader = DataLoader(CatDogDataset(train_files, image_dir, transform=train_transform),
                          batch_size=32, shuffle=True)
val_loader   = DataLoader(CatDogDataset(val_files, image_dir, transform=val_transform),
                          batch_size=32, shuffle=False)

## 3. Архитектура сети

Три свёрточных блока (Conv → ReLU → MaxPool): изображение `128×128` сжимается до `16×16` с 64 каналами, затем линейный слой на 2 класса.

In [ ]:
class CatsDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Linear(64 * 16 * 16, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = CatsDogCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print('Устройство:', device)

## 4. Обучение

In [ ]:
epochs = 5
for epoch in range(epochs):
    model.train()
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_acc = correct / total

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            v_correct += (outputs.argmax(1) == labels).sum().item()
            v_total += labels.size(0)
    val_acc = v_correct / v_total

    print(f'Эпоха {epoch+1}/{epochs} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%')

## 5. Предсказание на тесте и сабмит

`id` берём из имени файла (`123.jpg` → `123`).

In [ ]:
test_files = sorted([f for f in os.listdir(test_dir) if f.endswith('.jpg')])
test_loader = DataLoader(CatDogDataset(test_files, test_dir, is_test=True, transform=val_transform),
                         batch_size=32, shuffle=False)

model.eval()
predictions = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        predictions.extend(model(images).argmax(1).cpu().numpy())

image_ids = [int(f.split('.')[0]) for f in test_files]
submission = pd.DataFrame({'id': image_ids, 'label': predictions})
submission.to_csv('submission_cats_dogs.csv', index=False)
print('submission_cats_dogs.csv сохранён!')
submission.head()

## Итог

| Шаг | Что сделали |
|---|---|
| Данные | Картинки → `128×128` → тензоры, метка из имени файла |
| Аугментация | Случайное отражение на обучении |
| Модель | 3 свёрточных блока + линейный классификатор |
| Обучение | Adam, CrossEntropyLoss, 5 эпох |
| Метрика | Accuracy на валидации |

**Как улучшить:** предобученная сеть (ResNet, transfer learning), больше аугментаций, Dropout, больше эпох.